In [ ]:
# Establish connection to workspace

from azure.ai.ml import MLClient
from azure.identity import DefaultAzureCredential

ml_client = MLClient.from_config(
    credential=DefaultAzureCredential()
)

print(ml_client)

#### Pipeline
-  A pipeline cab be machine learning task, and each task contain multiple steps, and steps can be a separate component. so a pipeline consist of multiple components. Each component can be tested and registered separately.

- Lets create an training pipeline
1. First create a dataset and register as a data asset
2.  

In [ ]:
# 1. Create a data asset
data_path = "data/train"
dataset_name = "heart-dataset-train"


heart_dataset_train = Data(
    path=data_path,
    type=AssetTypes.URI_FOLDER,
    description="A training dataset for heart classification",
    name=dataset_name,
)
ml_client.data.create_or_update(heart_dataset_train)

In [ ]:
# Load the data asset
dataset_name = "heart-dataset-train"
heart_dataset_train = ml_client.data.get(name=dataset_name, label="latest")

In [ ]:
# create a pipeline using components prepare and train

from azure.ai.ml import MLClient, Input, load_component, Output
from azure.ai.ml.dsl import pipeline
from azure.ai.ml.constants import AssetTypes


prepare_data = load_component(source="components/prepare/prepare.yml")
train_xgb = load_component(source="components/train_xgb/train_xgb.yml")

@pipeline()
def uci_heart_classifier_trainer(input_data: Input(type=AssetTypes.URI_FOLDER)):
    prepared_data = prepare_data(data=input_data)
    trained_model = train_xgb(
        data=prepared_data.outputs.prepared_data,
        target_column="target",
        register_best_model=False,
        eval_size=0.3,
    )

    return {
        "model": trained_model.outputs.model,
        "evaluation_results": trained_model.outputs.evaluation_results,
        "transformations_output": prepared_data.outputs.transformations_output,
    }



#### Components
- A pipeline usually consist of multiple components.
- Testing each invidual component is a best practice.
- we can create a pipeline with one component only and run as a job.
- By default the output of the component is saved in default location
- we can customize at component level inside the pipeline and other way be can set at pipeline output level. 
- Internally both works same, store is a custom location provided

In [ ]:
# prepare_data component
# run individual components to test them
# output of the component will be stored in the default location

prepare_component = load_component(
    source="components/prepare/prepare.yml"
)

# Create a pipeline with only one component

@pipeline()
def uci_heart_classifier_prepare_data(input_data: Input(type=AssetTypes.URI_FOLDER)):
    prepared_data = prepare_data(data=input_data)
    
    return {
        "prepared_data" : prepared_data.outputs.prepared_data,
        "transformations_output": prepared_data.outputs.transformations_output,
    }


In [ ]:
# custom path at component/node level
# using custom path we can save output result to specific location
prepared_data_output_custom_path = "azureml://datastores/workspaceblobstore/paths/custom_path/${{name}}/"


@pipeline()
def uci_heart_classifier_prepare_data(input_data: Input(type=AssetTypes.URI_FOLDER)):
    prepared_data = prepare_data(data=input_data)
    
    #set a custom path for the output of the prepare_data component's "prepared_data" output # Node level
    prepared_data.outputs.prepared_data = Output(
    type="uri_folder", mode="rw_mount", path=prepared_data_output_custom_path)
    
    return {
        "prepared_data" : prepared_data.outputs.prepared_data,
        "transformations_output": prepared_data.outputs.transformations_output,
    }



In [ ]:
pipeline_job = uci_heart_classifier_prepare_data(
    Input(type="uri_folder", path=heart_dataset_train.id)
)

# custom path at pipeline level
transformations_output_custom_path = "azureml://datastores/workspaceblobstore/paths/custom_path/${{name}}/"
pipeline_job.outputs.transformations_output.path = transformations_output_custom_path

pipeline_job.settings.default_datastore = "workspaceblobstore"
pipeline_job.settings.default_compute = "batch-cluster"

In [ ]:
pipeline_job_run = ml_client.jobs.create_or_update(
    pipeline_job, experiment_name="uci-heart-train-pipeline2"
)
pipeline_job_run

In [ ]:
# Instead creating pipeline in python, we can load the pipeline from a yml file.

from azure.ai.ml import load_job, Input
from azure.ai.ml.constants import AssetTypes

pipeline_job = load_job(
    source="pipeline/pipeline.yml"
)


# Pass input dynamically
pipeline_job.inputs.input_data = Input(
    type=AssetTypes.URI_FOLDER,
    path=heart_dataset_train.id
)

# Submit pipeline
returned_job = ml_client.jobs.create_or_update(
    pipeline_job,
    experiment_name="uci-heart-experiment"
)

print(returned_job.name)

#### same job we can run in CLI using command:
`az ml job create \
    --file pipeline/pipeline.yml \
    --set inputs.input_data.path="azureml:heart-dataset-train:1" \
    --resource-group  "ysn-resource" \
    --workspace-name "mlw-basic-prod-202611131047"`